# Glacial Lake Polygons Post-processing

This notebook loads the shapefile generated from tiled model inference and:
1. Merges polygons that belong to the **same lake across multiple tiles**.
2. Removes **duplicate/overlapping polygons** caused by overlap in Planet imagery.

You only need to update the `INPUT_SHP` and `OUTPUT_SHP` paths in the cell below.

In [1]:
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from pathlib import Path

print("Geopandas version:", gpd.__version__)

Geopandas version: 1.1.1


In [2]:
# === USER INPUT ===
# Path to the shapefile produced by your inference pipeline
INPUT_SHP = r"D:\Thesis\glacial_lake_detection_thesis\data\inference\outputs\output_512\train_256_lakes\lakes_wo_class_imbalance\lakes\lakes.shp"  # <-- change this to your shapefile path

# Path where the cleaned / merged lakes shapefile will be saved
OUTPUT_SHP = r"D:\Thesis\glacial_lake_detection_thesis\data\inference\outputs\output_refined\train_256_lakes\lakes_wo_class_imbalance\lakes_merged_clean.shp"  # <-- change if you like

# Buffer distance in **meters** used to merge polygons that are touching/overlapping
# For PlanetScope (~3m resolution), 10–30 m is usually safe. You can tweak this.
BUFFER_M = 20

# Minimum lake area threshold in square meters.
# Anything smaller will be dropped as noise (set to 0 to keep everything).
MIN_AREA_M2 = 0


In [3]:
def merge_lake_polygons(input_shp: str,
                         output_shp: str,
                         buffer_m: float = 15,
                         min_area_m2: float = 0.0):
    """Load raw lake polygons from tiles, merge duplicates/overlaps, and save a cleaned shapefile.

    Steps:
    1. Read shapefile and fix invalid geometries.
    2. Reproject to a metric CRS (if needed) so buffering is in meters.
    3. Buffer -> dissolve -> negative buffer to merge touching/overlapping polygons.
    4. Explode MultiPolygons into individual polygons.
    5. Optionally remove polygons smaller than `min_area_m2`.
    6. Assign a new `lake_id` and save to disk.
    """

    input_path = Path(input_shp)
    if not input_path.exists():
        raise FileNotFoundError(f"Input shapefile not found: {input_path}")

    print(f"Loading lakes from: {input_path}")
    gdf = gpd.read_file(input_path)
    print("Original features:", len(gdf))

    # Fix invalid geometries (self-intersections etc.)
    print("Fixing invalid geometries using buffer(0)...")
    gdf["geometry"] = gdf.geometry.buffer(0)

    original_crs = gdf.crs
    print("Original CRS:", original_crs)

    # If CRS is geographic (degrees), reproject to suitable UTM so buffer is in meters
    if original_crs is not None and getattr(original_crs, 'is_geographic', False):
        print("Reprojecting to estimated UTM CRS for metric buffering...")
        metric_crs = gdf.estimate_utm_crs()
        gdf = gdf.to_crs(metric_crs)
        print("Metric CRS:", metric_crs)
    else:
        print("Assuming current CRS is already metric (e.g., UTM).")

    # Create a dummy dissolve field so all features are in one group
    gdf["_dissolve_id"] = 1

    print(f"Buffering geometries by +{buffer_m} m to connect touching/nearby polygons...")
    gdf_buffered = gdf.copy()
    gdf_buffered["geometry"] = gdf_buffered.geometry.buffer(buffer_m)

    print("Dissolving buffered geometries into merged lakes...")
    dissolved = gdf_buffered.dissolve(by="_dissolve_id")

    # Remove helper column if present
    if "_dissolve_id" in dissolved.columns:
        dissolved = dissolved.drop(columns=["_dissolve_id"])

    print(f"Shrinking geometries back by -{buffer_m} m to approximate original shapes...")
    dissolved["geometry"] = dissolved.geometry.buffer(-buffer_m)

    # Explode MultiPolygons into individual rows
    print("Exploding MultiPolygons into individual Polygon features...")
    dissolved = dissolved.explode(index_parts=False).reset_index(drop=True)

    # Remove empty geometries
    dissolved = dissolved[~dissolved.geometry.is_empty]
    dissolved = dissolved[~dissolved.geometry.isna()]

    # Compute area in m^2 (since we're in a metric CRS)
    print("Computing polygon areas (m^2)...")
    dissolved["area_m2"] = dissolved.geometry.area

    if min_area_m2 > 0:
        print(f"Dropping polygons smaller than {min_area_m2} m^2...")
        before = len(dissolved)
        dissolved = dissolved[dissolved["area_m2"] >= min_area_m2]
        after = len(dissolved)
        print(f"Removed {before - after} tiny polygons.")

    # Assign a new unique lake_id
    dissolved = dissolved.reset_index(drop=True)
    dissolved["lake_id"] = range(1, len(dissolved) + 1)

    print("Final number of merged lakes:", len(dissolved))

    # Reproject back to original CRS if it existed and was geographic
    if original_crs is not None and getattr(original_crs, 'is_geographic', False):
        print("Reprojecting merged lakes back to original CRS...")
        dissolved = dissolved.to_crs(original_crs)

    output_path = Path(output_shp)
    dissolved.to_file(output_path)
    print(f"Saved cleaned lakes to: {output_path.resolve()}")

    return dissolved


In [4]:
# === RUN THE MERGING / CLEANING ===
merged_gdf = merge_lake_polygons(
    input_shp=INPUT_SHP,
    output_shp=OUTPUT_SHP,
    buffer_m=BUFFER_M,
    min_area_m2=MIN_AREA_M2,
)

merged_gdf.head()

Loading lakes from: D:\Thesis\glacial_lake_detection_thesis\data\inference\outputs\output_512\train_256_lakes\lakes_wo_class_imbalance\lakes\lakes.shp
Original features: 413509
Fixing invalid geometries using buffer(0)...
Original CRS: EPSG:4326
Reprojecting to estimated UTM CRS for metric buffering...
Metric CRS: EPSG:32643
Buffering geometries by +20 m to connect touching/nearby polygons...
Dissolving buffered geometries into merged lakes...
Shrinking geometries back by -20 m to approximate original shapes...
Exploding MultiPolygons into individual Polygon features...
Computing polygon areas (m^2)...
Final number of merged lakes: 241884
Reprojecting merged lakes back to original CRS...


c:\Users\gg\anaconda3\envs\thesis\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 271888727.52576554 of field area_m2 of feature 93276 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(


Saved cleaned lakes to: D:\Thesis\glacial_lake_detection_thesis\data\inference\outputs\output_refined\train_256_lakes\lakes_wo_class_imbalance\lakes_merged_clean.shp


,source_img,geometry,area_m2,lake_id
0,20250902_060832_78_24da_3B_Visual_clip_reproje...,"POLYGON ((77.53334 35.35147, 77.53334 35.35147...",129.802709,1
1,20250902_060832_78_24da_3B_Visual_clip_reproje...,"POLYGON ((77.52723 35.3498, 77.52723 35.3498, ...",727.796854,2
2,20250902_060832_78_24da_3B_Visual_clip_reproje...,"POLYGON ((77.52574 35.35296, 77.52575 35.35297...",200.965810,3
3,20250902_060832_78_24da_3B_Visual_clip_reproje...,"POLYGON ((77.51864 35.33296, 77.51865 35.33296...",19.359333,4
4,20250902_060832_78_24da_3B_Visual_clip_reproje...,"POLYGON ((77.51687 35.336, 77.51688 35.33601, ...",1620.291756,5
